In [1]:
# import libraries
import os
import pandas as pd

import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# Define lambda functions for path construction
get_data_path = lambda folders, fname: os.path.normpath(os.environ['DRIVE_PATH'] + '/' + '/'.join(folders) + '/' + fname)

# Construct the file paths
file_path_tissue_analysis_hits = get_data_path(['Sanger', 'Vicky_GI_analysis', 'tissue_analysis', 'Archive'], 'vicky_hits_updated.csv')
file_path_tissue_analysis_gi_scores = get_data_path(['Sanger', 'Vicky_GI_analysis', 'tissue_analysis', 'Archive'], 'vicky_scores_updated.csv')

# Outputs
figure_folder = ['Sanger', 'Vicky_GI_analysis', 'tissue_analysis', 'Archive', 'figures']
file_path_tissue_associations = get_data_path(['Sanger', 'Vicky_GI_analysis', 'tissue_analysis', 'Archive'], 'tissue_associations.csv')

In [3]:
# read in the data
all_bimodal_genes = set()
with open(file_path_tissue_analysis_hits, "r") as f:
    for l in f :
        all_bimodal_genes.add(l.split(',')[0])

In [4]:
len(all_bimodal_genes)

131

In [5]:
screen = pd.read_csv(file_path_tissue_analysis_gi_scores,index_col=0)
screen.shape

(27, 473)

In [6]:
# We'll perform ANOVA for each perturbation and collect p-values
p_values = []
perturbations = []

for column in all_bimodal_genes.intersection(screen.columns):
    if column not in ['Cell_line', 'Tissue']:
        # Performing ANOVA
        mod = sm.formula.ols(f'{column} ~ C(Tissue)', data=screen).fit()
        anova_table = sm.stats.anova_lm(mod, typ=2)
        p_values.append(anova_table['PR(>F)'].iloc[0])
        perturbations.append(column)

# Applying Benjamini-Hochberg correction
reject, pvals_corrected, _, _ = multipletests(p_values, alpha=0.1, method='fdr_bh')
# Results after correction
results = pd.DataFrame({
    'Perturbation': perturbations,
    'P-Value': p_values,
    'Adjusted P-Value': pvals_corrected,
    'Reject Null': reject
})


# Follow-up with post-hoc tests if needed (for significant results)
for index, row in results[results['Reject Null']].iterrows():
    perturbation = row['Perturbation']
    posthoc = pairwise_tukeyhsd(screen[perturbation], screen['Tissue'], alpha=0.05)
    
    if row["Adjusted P-Value"] < 0.1  :
        print(f'Results for {perturbation}:')
        print(posthoc)


Results for PDS5A_PDS5B:
  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
 group1   group2  meandiff p-adj   lower  upper  reject
-------------------------------------------------------
    LUNG PANCREAS  -0.4347 0.0161 -0.7953 -0.074   True
    LUNG     SKIN   0.0817 0.8484 -0.2906 0.4541  False
PANCREAS     SKIN   0.5164 0.0067   0.135 0.8978   True
-------------------------------------------------------
Results for CLDN4_TIMM10:
  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1   group2  meandiff p-adj   lower   upper  reject
--------------------------------------------------------
    LUNG PANCREAS  -0.1355  0.152 -0.3108  0.0398  False
    LUNG     SKIN  -0.2332 0.0099 -0.4141 -0.0523   True
PANCREAS     SKIN  -0.0977    0.4 -0.2831  0.0876  False
--------------------------------------------------------
Results for STX7_STX12:
  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1   group2  meandiff p-adj   lower   upper  reject
--------------------

In [8]:
# Filter for significant results
significant_perturbations = results[results['Reject Null']]['Perturbation']
palette = {'SKIN': '#F8D687', 'PANCREAS': '#649B92', 'LUNG': '#84B0D1'}

for perturbation in significant_perturbations:
    plt.figure(figsize=(6, 6))
    ax = sns.boxplot(x='Tissue', y=perturbation, data=screen, palette = palette, showfliers=False)
    sns.swarmplot(x='Tissue', y=perturbation, data=screen, color='black', size=4)
    plt.title(f'{perturbation}')
    plt.ylabel('Score')
    plt.xlabel('Cancer Type')
    labels = [label.get_text().replace('SKIN', 'Melanoma').replace('PANCREAS', 'Pancreatic').replace("LUNG", "Lung") for label in ax.get_xticklabels()]
    ax.set_xticklabels(labels)
    plt.savefig(f'TissueFigs/{perturbation}_box_plot.pdf', format='pdf')
    plt.close()

In [7]:
# Filter for significant results
significant_perturbations = results[results['Reject Null']]['Perturbation']
palette = {'SKIN': '#F8D687', 'PANCREAS': '#649B92', 'LUNG': '#84B0D1'}

for perturbation in significant_perturbations:
    plt.figure(figsize=(3, 2))

    ax = sns.boxplot(x='Tissue', y=perturbation, data=screen, hue='Tissue', palette=palette, showfliers=False, dodge=False, boxprops=dict(alpha=.6))
    sns.swarmplot(x='Tissue', y=perturbation, data=screen, color='black', size=2.5)

    ax.set_xticks(ax.get_xticks())
    labels = [label.replace('SKIN', 'Melanoma').replace('PANCREAS', 'Pancreas').replace('LUNG', 'Lung NSCLC') for label in screen['Tissue'].unique()]
    ax.set_xticklabels(labels)

    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=5)

    ax.set_xlabel('')
    ax.set_ylabel('GI Score', fontsize=8)
    ax.set_title(f'{perturbation}', fontsize=8)

    plt.tight_layout()
    #plt.subplots_adjust(top=0.80, bottom=0.01, left=0.2, right=0.85, hspace=0.05, wspace=0.05)
    plt.savefig(get_data_path(figure_folder, f'{perturbation}_box_plot.png'), dpi=500)
    plt.close()


In [9]:
results.to_csv(file_path_tissue_associations)